#  Experiments

## Imports & Setup

In [2]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import xgboost as xgb
import lightgbm as lgb
import catboost as cb
import optuna

from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

import mlflow
import os
import joblib
from pathlib import Path
from dotenv import load_dotenv
from pandas.tseries.holiday import USFederalHolidayCalendar

optuna.logging.set_verbosity(optuna.logging.WARNING)

load_dotenv()

MLFLOW_TRACKING_URI = os.environ["MLFLOW_TRACKING_URI"]
MLFLOW_TRACKING_USERNAME = os.environ["MLFLOW_TRACKING_USERNAME"]
MLFLOW_TRACKING_PASSWORD = os.environ["MLFLOW_TRACKING_PASSWORD"]

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment("electricity-load-forecast-production")

/teamspace/studios/this_studio/electricity-distribution-forecast/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<Experiment: artifact_location='mlflow-artifacts:/80f4bff9ab5d44a1b1bdb82527644551', creation_time=1786487731372, effective_trace_archival_retention=None, experiment_id='2', last_update_time=1786487731372, lifecycle_stage='active', name='electricity-load-forecast-production', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

## Load Data

In [3]:
df = pd.read_parquet("../data/interim/df_core_features.parquet")
df = df.sort_values("timestamp").reset_index(drop=True)
print("Df shape:", df.shape)
df.tail(1)

Df shape: (92709, 16)


,timestamp,timestamp_central,actual_load_mw,temp_c,humidity_pct,precip_mm,tmax,tmin,tavg,hour,dayofweek,month,is_weekend,is_holiday,load_lag_24,load_lag_168
92708,2026-08-05 23:00:00+00:00,2026-08-05 18:00:00-05:00,15006.54,35.48255,35.336102,0.114556,36.452254,25.396689,30.708715,18,2,8,0,0,15250.65,15183.67


## Chronological Split

In [4]:
PURGE_DAYS = 7

TEST_START_CENTRAL = pd.Timestamp("2025-08-13 00:00:00", tz="America/Chicago")
TEST_START = TEST_START_CENTRAL.tz_convert("UTC")
TRAIN_END  = TEST_START - pd.Timedelta(days=PURGE_DAYS)

train = df[df["timestamp"] <= TRAIN_END].reset_index(drop=True)
test  = df[df["timestamp"] >= TEST_START].reset_index(drop=True)

print(f"Train: {len(train):,} rows | {train['timestamp'].min()} -> {train['timestamp'].max()}")
print(f"Purge gap: {TRAIN_END} -> {TEST_START}  ({PURGE_DAYS} days)")
print(f"Test:  {len(test):,} rows | {test['timestamp'].min()} -> {test['timestamp'].max()}")

Train: 83,955 rows | 2016-01-08 00:00:00+00:00 -> 2025-08-06 05:00:00+00:00
Purge gap: 2025-08-06 05:00:00+00:00 -> 2025-08-13 05:00:00+00:00  (7 days)
Test:  8,587 rows | 2025-08-13 05:00:00+00:00 -> 2026-08-05 23:00:00+00:00


## Cross-Validation Folds

In [5]:
fold_boundaries = [
    ("2016-01-08 00:00:00+00:00", "2020-07-30 05:00:00+00:00",
     "2020-08-06 06:00:00+00:00", "2021-08-06 05:00:00+00:00"),

    ("2016-01-08 00:00:00+00:00", "2021-07-30 05:00:00+00:00",
     "2021-08-06 06:00:00+00:00", "2022-08-06 05:00:00+00:00"),

    ("2016-01-08 00:00:00+00:00", "2022-07-30 05:00:00+00:00",
     "2022-08-06 06:00:00+00:00", "2023-08-06 05:00:00+00:00"),

    ("2016-01-08 00:00:00+00:00", "2023-07-30 05:00:00+00:00",
     "2023-08-06 06:00:00+00:00", "2024-08-06 05:00:00+00:00"),

    ("2016-01-08 00:00:00+00:00", "2024-07-30 05:00:00+00:00",
     "2024-08-06 06:00:00+00:00", "2025-08-06 05:00:00+00:00"),
]

TARGET = "actual_load_mw"
FEATURES = ["temp_c", "humidity_pct", "precip_mm", "tmax", "tmin", "tavg",
            "hour", "dayofweek", "month", "is_weekend", "is_holiday",
            "load_lag_24", "load_lag_168"]

## Feature Engineering

In [6]:
df["trend_idx"] = (df["timestamp"] - df["timestamp"].min()).dt.total_seconds() / (3600 * 24 * 365)

df["hour_x_temp"] = df["hour"] * df["temp_c"]
df["temp_c_roll_std_72"] = df["temp_c"].rolling(window=72, min_periods=24).std()

df["temp_c_lag_24"] = df["temp_c"].shift(24)
df["temp_change_vs_lag24"] = df["temp_c"] - df["temp_c_lag_24"]

train = df[df["timestamp"] <= TRAIN_END].reset_index(drop=True)
test  = df[df["timestamp"] >= TEST_START].reset_index(drop=True)

FEATURES_V2 = [
    "temp_c", "humidity_pct", "precip_mm", "tmax", "tmin", "tavg",
    "hour", "dayofweek", "month", "is_weekend", "is_holiday",
    "load_lag_24", "load_lag_168", "hour_x_temp", "temp_c_roll_std_72",
    "is_extreme_heat_event", "is_extreme_cold_event", "is_holiday_x_extreme",
]

FEATURES_V3 = FEATURES_V2 + ["temp_change_vs_lag24", "is_high_precip_event"]

cal = USFederalHolidayCalendar()
start = train["timestamp_central"].min().tz_localize(None)
end   = train["timestamp_central"].max().tz_localize(None)
holiday_names = cal.holidays(start=start, end=end, return_name=True)

for part in (train, test):
    central_date = part["timestamp_central"].dt.normalize().dt.tz_localize(None)
    part["holiday_name"] = central_date.map(holiday_names)

def add_v3_fold_features(fold_train, fold_val):
    heat_thresh   = fold_train["tmax"].quantile(0.95)
    cold_thresh   = fold_train["tmin"].quantile(0.05)
    precip_thresh = fold_train["precip_mm"].quantile(0.95)
    for part in (fold_train, fold_val):
        part["is_extreme_heat_event"] = (part["tmax"] > heat_thresh).astype(int)
        part["is_extreme_cold_event"] = (part["tmin"] < cold_thresh).astype(int)
        part["is_holiday_x_extreme"] = (
            part["is_holiday"] * (part["is_extreme_heat_event"] | part["is_extreme_cold_event"])
        ).astype(int)
        part["is_high_precip_event"] = (part["precip_mm"] > precip_thresh).astype(int)
    return fold_train, fold_val

## Baseline Models Constant & Persistence

In [7]:
oof_mean = np.full(len(train), np.nan)
oof_persistence = np.full(len(train), np.nan)

for fold_id, (tr_start, tr_end, val_start, val_end) in enumerate(fold_boundaries, start=1):
    train_mask = (train["timestamp"] >= tr_start) & (train["timestamp"] <= tr_end)
    val_mask   = (train["timestamp"] >= val_start) & (train["timestamp"] <= val_end)

    y_train = train.loc[train_mask, TARGET]
    y_val   = train.loc[val_mask, TARGET]

    mean_pred = np.full(val_mask.sum(), y_train.mean())
    oof_mean[val_mask.values] = mean_pred

    persistence_pred = train.loc[val_mask, "load_lag_24"].values
    oof_persistence[val_mask.values] = persistence_pred

    rmse_mean = mean_squared_error(y_val, mean_pred) ** 0.5
    rmse_pers = mean_squared_error(y_val, persistence_pred) ** 0.5
    print(f"Fold {fold_id}  |  Constant RMSE: {rmse_mean:.2f}  |  Persistence RMSE: {rmse_pers:.2f}")

for name, oof in [("Constant", oof_mean), ("Persistence", oof_persistence)]:
    valid = ~np.isnan(oof)
    rmse = mean_squared_error(train.loc[valid, TARGET], oof[valid]) ** 0.5
    print(f"{name} OOF RMSE: {rmse:.2f}")

Fold 1  |  Constant RMSE: 1807.06  |  Persistence RMSE: 772.98
Fold 2  |  Constant RMSE: 2331.90  |  Persistence RMSE: 811.14
Fold 3  |  Constant RMSE: 2390.09  |  Persistence RMSE: 804.75
Fold 4  |  Constant RMSE: 2585.58  |  Persistence RMSE: 800.56
Fold 5  |  Constant RMSE: 2592.82  |  Persistence RMSE: 843.63
Constant OOF RMSE: 2359.10
Persistence OOF RMSE: 806.93


## Linear Regression

In [8]:
oof_lr = np.full(len(train), np.nan)

for fold_id, (tr_start, tr_end, val_start, val_end) in enumerate(fold_boundaries, start=1):
    train_mask = (train["timestamp"] >= tr_start) & (train["timestamp"] <= tr_end)
    val_mask   = (train["timestamp"] >= val_start) & (train["timestamp"] <= val_end)

    X_train, y_train = train.loc[train_mask, FEATURES], train.loc[train_mask, TARGET]
    X_val,   y_val   = train.loc[val_mask,   FEATURES], train.loc[val_mask,   TARGET]

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_val_s   = scaler.transform(X_val)

    lr_model = LinearRegression()
    lr_model.fit(X_train_s, y_train)

    val_preds = lr_model.predict(X_val_s)
    oof_lr[val_mask.values] = val_preds

    rmse = mean_squared_error(y_val, val_preds) ** 0.5
    print(f"Fold {fold_id} RMSE: {rmse:.2f}")

# -------------------------
# Overall Metrics Evaluation
# -------------------------
valid_lr = ~np.isnan(oof_lr)
y_true_valid = train.loc[valid_lr, TARGET]
y_pred_valid = oof_lr[valid_lr]

oof_rmse_lr = mean_squared_error(y_true_valid, y_pred_valid) ** 0.5
oof_mape_lr = mean_absolute_percentage_error(y_true_valid, y_pred_valid) * 100

print("-" * 40)
print(f"Linear Model, CV OOF RMSE: {oof_rmse_lr:.2f}")
print(f"Linear Model, CV OOF MAPE: {oof_mape_lr:.2f}%")

# -------------------------
# MLflow Logging
# -------------------------
with mlflow.start_run(run_name="Baseline_Linear_Regression"):
    # 1. Log parameters
    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_param("features", "raw_features")
    mlflow.log_param("cv_strategy", "5-fold-expanding")
    
    # 2. Log metrics
    mlflow.log_metric("cv_oof_rmse", oof_rmse_lr)
    mlflow.log_metric("cv_oof_mape", oof_mape_lr)
    
    print("Successfully logged params and metrics to MLflow (no model artifacts saved).")

Fold 1 RMSE: 740.43
Fold 2 RMSE: 766.36
Fold 3 RMSE: 747.31
Fold 4 RMSE: 761.36
Fold 5 RMSE: 795.28
----------------------------------------
Linear Model, CV OOF RMSE: 762.39
Linear Model, CV OOF MAPE: 6.47%


Successfully logged params and metrics to MLflow (no model artifacts saved).
🏃 View run Baseline_Linear_Regression at: https://dagshub.com/dunnioluajayi/electricity-distribution-forecast.mlflow/#/experiments/2/runs/61b0357dde3742419aa6f6cc71f96e97
🧪 View experiment at: https://dagshub.com/dunnioluajayi/electricity-distribution-forecast.mlflow/#/experiments/2


## XGBoost (Baseline)

In [9]:

oof_xgb = np.full(len(train), np.nan)

print("Starting XGBoost 5-Fold Cross-Validation...\n")

for fold_id, (tr_start, tr_end, val_start, val_end) in enumerate(fold_boundaries, start=1):
    train_mask = (train["timestamp"] >= tr_start) & (train["timestamp"] <= tr_end)
    val_mask   = (train["timestamp"] >= val_start) & (train["timestamp"] <= val_end)

    X_train, y_train = train.loc[train_mask, FEATURES], train.loc[train_mask, TARGET]
    X_val,   y_val   = train.loc[val_mask,   FEATURES], train.loc[val_mask,   TARGET]

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_val_s   = scaler.transform(X_val)

    xgb_baseline = xgb.XGBRegressor(
        n_estimators=500, learning_rate=0.05, max_depth=6,
        random_state=42, n_jobs=-1
    )
    xgb_baseline.fit(X_train_s, y_train)

    val_preds = xgb_baseline.predict(X_val_s)
    oof_xgb[val_mask.values] = val_preds

    rmse = mean_squared_error(y_val, val_preds) ** 0.5
    print(f"Fold {fold_id} RMSE: {rmse:.2f}")

# -------------------------
# Overall Metrics Evaluation
# -------------------------
valid_xgb_baseline = ~np.isnan(oof_xgb)
y_true_valid = train.loc[valid_xgb_baseline, TARGET]
y_pred_valid = oof_xgb[valid_xgb_baseline]

oof_rmse_xgb = mean_squared_error(y_true_valid, y_pred_valid) ** 0.5
oof_mape_xgb = mean_absolute_percentage_error(y_true_valid, y_pred_valid) * 100

print("-" * 40)
print(f"XGBoost Baseline, CV OOF RMSE: {oof_rmse_xgb:.2f}")
print(f"XGBoost Baseline, CV OOF MAPE: {oof_mape_xgb:.2f}%")

# -------------------------
# MLflow Logging
# -------------------------
with mlflow.start_run(run_name="Baseline_XGBoost_Raw"):
    # 1. Log general parameters
    mlflow.log_param("model_type", "XGBoost")
    mlflow.log_param("features", "raw_features")
    mlflow.log_param("cv_strategy", "5-fold-expanding")
    
    # 2. Log specific hyperparameters
    mlflow.log_param("n_estimators", 500)
    mlflow.log_param("learning_rate", 0.05)
    mlflow.log_param("max_depth", 6)
    
    # 3. Log metrics
    mlflow.log_metric("cv_oof_rmse", oof_rmse_xgb)
    mlflow.log_metric("cv_oof_mape", oof_mape_xgb)
    
    print("Successfully logged params and metrics to MLflow (no model artifacts saved).")

Starting XGBoost 5-Fold Cross-Validation...

Fold 1 RMSE: 456.26
Fold 2 RMSE: 616.07
Fold 3 RMSE: 757.18
Fold 4 RMSE: 692.94
Fold 5 RMSE: 678.91
----------------------------------------
XGBoost Baseline, CV OOF RMSE: 648.43
XGBoost Baseline, CV OOF MAPE: 6.32%
Successfully logged params and metrics to MLflow (no model artifacts saved).
🏃 View run Baseline_XGBoost_Raw at: https://dagshub.com/dunnioluajayi/electricity-distribution-forecast.mlflow/#/experiments/2/runs/05e5528494834628b2389eb0154674ae
🧪 View experiment at: https://dagshub.com/dunnioluajayi/electricity-distribution-forecast.mlflow/#/experiments/2


## Trend + XGBoost-on-Residual (v1)


In [10]:
df["trend_idx"] = (df["timestamp"] - df["timestamp"].min()).dt.total_seconds() / (3600 * 24 * 365)
train = df[df["timestamp"] <= TRAIN_END].reset_index(drop=True)
test  = df[df["timestamp"] >= TEST_START].reset_index(drop=True)

fold_results = {}
oof_trend_xgb = np.full(len(train), np.nan)

print("Starting Trend + XGBoost (v1) 5-Fold Cross-Validation...\n")

# Cross-Validation Loop
for fold_id, (tr_start, tr_end, val_start, val_end) in enumerate(fold_boundaries, start=1):
    train_mask = (train["timestamp"] >= tr_start) & (train["timestamp"] <= tr_end)
    val_mask   = (train["timestamp"] >= val_start) & (train["timestamp"] <= val_end)

    X_train, y_train = train.loc[train_mask, FEATURES], train.loc[train_mask, TARGET]
    X_val,   y_val    = train.loc[val_mask,   FEATURES], train.loc[val_mask,   TARGET]
    trend_train = train.loc[train_mask, ["trend_idx"]]
    trend_val   = train.loc[val_mask,   ["trend_idx"]]

    # Fit linear trend model
    trend_model = LinearRegression()
    trend_model.fit(trend_train, y_train)
    trend_pred_train = trend_model.predict(trend_train)
    trend_pred_val   = trend_model.predict(trend_val)

    # Calculate residual target for XGBoost
    resid_train = y_train - trend_pred_train

    # Fit XGBoost on trend residuals
    xgb_model = xgb.XGBRegressor(
        n_estimators=500, learning_rate=0.05, max_depth=6,
        random_state=42, n_jobs=-1
    )
    xgb_model.fit(X_train, resid_train)
    resid_pred_val = xgb_model.predict(X_val)

    # Recombine linear trend + non-linear residual predictions
    final_pred = trend_pred_val + resid_pred_val
    residual = y_val.values - final_pred

    fold_results[fold_id] = {
        "trend_model": trend_model, "xgb_model": xgb_model,
        "X_train": X_train, "X_val": X_val, "y_val": y_val,
        "final_pred": final_pred, "residual": residual,
        "timestamps": train.loc[val_mask, "timestamp_central"].values,
        "train_mask": train_mask, "val_mask": val_mask,
    }

    oof_trend_xgb[val_mask.values] = final_pred
    rmse = mean_squared_error(y_val, final_pred) ** 0.5
    print(f"Fold {fold_id} RMSE: {rmse:.2f}")

# Overall Metrics Evaluation
valid_trend_xgb = ~np.isnan(oof_trend_xgb)
y_true_valid = train.loc[valid_trend_xgb, TARGET]
y_pred_valid = oof_trend_xgb[valid_trend_xgb]

oof_rmse_trend_xgb = mean_squared_error(y_true_valid, y_pred_valid) ** 0.5
oof_mape_trend_xgb = mean_absolute_percentage_error(y_true_valid, y_pred_valid) * 100

print("-" * 40)
print(f"Trend + XGB (v1), CV OOF RMSE: {oof_rmse_trend_xgb:.2f}")
print(f"Trend + XGB (v1), CV OOF MAPE: {oof_mape_trend_xgb:.2f}%")

#  MLflow Logging
with mlflow.start_run(run_name="Trend_XGB_v1"):
    # Log general parameters & experiment strategy
    mlflow.log_param("model_type", "LinearRegression_Trend + XGBoost_Residual")
    mlflow.log_param("experiment_version", "v1")
    mlflow.log_param("detrended", True)
    mlflow.log_param("features", "raw_features + trend_idx")
    mlflow.log_param("cv_strategy", "5-fold-expanding")
    
    # Log XGBoost hyperparameters
    mlflow.log_param("n_estimators", 500)
    mlflow.log_param("learning_rate", 0.05)
    mlflow.log_param("max_depth", 6)
    
    # Log metrics
    mlflow.log_metric("cv_oof_rmse", oof_rmse_trend_xgb)
    mlflow.log_metric("cv_oof_mape", oof_mape_trend_xgb)
    
    print("Successfully logged params and metrics to MLflow (no model artifacts saved).")


Starting Trend + XGBoost (v1) 5-Fold Cross-Validation...



Fold 1 RMSE: 409.12
Fold 2 RMSE: 417.79
Fold 3 RMSE: 442.66
Fold 4 RMSE: 475.68
Fold 5 RMSE: 412.68
----------------------------------------
Trend + XGB (v1), CV OOF RMSE: 432.33
Trend + XGB (v1), CV OOF MAPE: 3.69%
Successfully logged params and metrics to MLflow (no model artifacts saved).
🏃 View run Trend_XGB_v1 at: https://dagshub.com/dunnioluajayi/electricity-distribution-forecast.mlflow/#/experiments/2/runs/5d0c677bfbe64fca8048aa18ec1d0f9c
🧪 View experiment at: https://dagshub.com/dunnioluajayi/electricity-distribution-forecast.mlflow/#/experiments/2


## Trend + XGBoost + Extreme-Event Features (v2)


In [11]:
df = df.sort_values("timestamp").reset_index(drop=True)

df["hour_x_temp"] = df["hour"] * df["temp_c"]
df["temp_c_roll_std_72"] = df["temp_c"].rolling(window=72, min_periods=24).std()

train = df[df["timestamp"] <= TRAIN_END].reset_index(drop=True)
test  = df[df["timestamp"] >= TEST_START].reset_index(drop=True)

FEATURES_V2 = [
    "temp_c", "humidity_pct", "precip_mm", "tmax", "tmin", "tavg",
    "hour", "dayofweek", "month", "is_weekend", "is_holiday",
    "load_lag_24", "load_lag_168", "hour_x_temp", "temp_c_roll_std_72",
    "is_extreme_heat_event", "is_extreme_cold_event", "is_holiday_x_extreme",
]

fold_results_v2 = {}
oof_trend_xgb_v2 = np.full(len(train), np.nan)

print("Starting Trend + XGBoost + Interactions (v2) 5-Fold Cross-Validation...\n")

for fold_id, (tr_start, tr_end, val_start, val_end) in enumerate(fold_boundaries, start=1):
    train_mask = (train["timestamp"] >= tr_start) & (train["timestamp"] <= tr_end)
    val_mask   = (train["timestamp"] >= val_start) & (train["timestamp"] <= val_end)

    fold_train = train.loc[train_mask].copy()
    fold_val   = train.loc[val_mask].copy()

    # Thresholds derived strictly from THIS fold's training data to prevent leakage
    heat_thresh = fold_train["tmax"].quantile(0.95)
    cold_thresh = fold_train["tmin"].quantile(0.05)

    for part in (fold_train, fold_val):
        part["is_extreme_heat_event"] = (part["tmax"] > heat_thresh).astype(int)
        part["is_extreme_cold_event"] = (part["tmin"] < cold_thresh).astype(int)
        part["is_holiday_x_extreme"] = (
            part["is_holiday"] * (part["is_extreme_heat_event"] | part["is_extreme_cold_event"])
        ).astype(int)

    X_train, y_train = fold_train[FEATURES_V2], fold_train[TARGET]
    X_val,   y_val    = fold_val[FEATURES_V2],   fold_val[TARGET]
    trend_train = fold_train[["trend_idx"]]
    trend_val   = fold_val[["trend_idx"]]

    # Linear trend model
    trend_model = LinearRegression()
    trend_model.fit(trend_train, y_train)
    trend_pred_train = trend_model.predict(trend_train)
    trend_pred_val   = trend_model.predict(trend_val)

    # Residual target for XGBoost
    resid_train = y_train - trend_pred_train

    xgb_model = xgb.XGBRegressor(
        n_estimators=500, learning_rate=0.05, max_depth=6,
        random_state=42, n_jobs=-1
    )
    xgb_model.fit(X_train, resid_train)
    resid_pred_val = xgb_model.predict(X_val)

    final_pred = trend_pred_val + resid_pred_val
    residual = y_val.values - final_pred

    fold_results_v2[fold_id] = {
        "xgb_model": xgb_model, "X_val": X_val, "y_val": y_val,
        "final_pred": final_pred, "residual": residual,
        "timestamps": fold_val["timestamp_central"].values,
        "heat_thresh": heat_thresh, "cold_thresh": cold_thresh,
    }

    oof_trend_xgb_v2[val_mask.values] = final_pred
    rmse = mean_squared_error(y_val, final_pred) ** 0.5
    print(f"Fold {fold_id} RMSE: {rmse:.2f}  (heat>{heat_thresh:.1f}°C, cold<{cold_thresh:.1f}°C)")

# -------------------------
# Overall Metrics Evaluation
# -------------------------
valid_v2 = ~np.isnan(oof_trend_xgb_v2)
y_true_v2 = train.loc[valid_v2, TARGET]
y_pred_v2 = oof_trend_xgb_v2[valid_v2]

oof_rmse_v2 = mean_squared_error(y_true_v2, y_pred_v2) ** 0.5
oof_mape_v2 = mean_absolute_percentage_error(y_true_v2, y_pred_v2) * 100

print("-" * 40)
print(f"Trend + XGB + interactions (v2), CV OOF RMSE: {oof_rmse_v2:.2f}   (v1: {oof_rmse_trend_xgb:.2f})")
print(f"Trend + XGB + interactions (v2), CV OOF MAPE: {oof_mape_v2:.2f}%")

# -------------------------
# MLflow Logging
# -------------------------
with mlflow.start_run(run_name="Trend_XGB_v2"):
    # Log general parameters & experiment strategy
    mlflow.log_param("model_type", "LinearRegression_Trend + XGBoost_Residual")
    mlflow.log_param("experiment_version", "v2")
    mlflow.log_param("detrended", True)
    mlflow.log_param("features", "FEATURES_V2 (Extreme events & holiday interactions)")
    mlflow.log_param("cv_strategy", "5-fold-expanding")
    
    # Log fold feature-engineering metadata
    mlflow.log_param("extreme_heat_quantile", 0.95)
    mlflow.log_param("extreme_cold_quantile", 0.05)
    
    # Log XGBoost hyperparameters
    mlflow.log_param("n_estimators", 500)
    mlflow.log_param("learning_rate", 0.05)
    mlflow.log_param("max_depth", 6)
    
    # Log metrics
    mlflow.log_metric("cv_oof_rmse", oof_rmse_v2)
    mlflow.log_metric("cv_oof_mape", oof_mape_v2)
    
    print("Successfully logged params and metrics to MLflow (no model artifacts saved).")


Starting Trend + XGBoost + Interactions (v2) 5-Fold Cross-Validation...



Fold 1 RMSE: 413.24  (heat>37.1°C, cold<3.3°C)
Fold 2 RMSE: 416.67  (heat>37.1°C, cold<2.9°C)
Fold 3 RMSE: 420.20  (heat>37.2°C, cold<2.6°C)
Fold 4 RMSE: 461.34  (heat>37.5°C, cold<2.6°C)
Fold 5 RMSE: 403.97  (heat>37.6°C, cold<2.7°C)
----------------------------------------
Trend + XGB + interactions (v2), CV OOF RMSE: 423.57   (v1: 432.33)
Trend + XGB + interactions (v2), CV OOF MAPE: 3.66%
Successfully logged params and metrics to MLflow (no model artifacts saved).
🏃 View run Trend_XGB_v2 at: https://dagshub.com/dunnioluajayi/electricity-distribution-forecast.mlflow/#/experiments/2/runs/b66180f603564d359b5761c07912d0a0
🧪 View experiment at: https://dagshub.com/dunnioluajayi/electricity-distribution-forecast.mlflow/#/experiments/2


## XGBoost (v3 features)

In [12]:
fold_results_v3 = {}
oof_trend_xgb_v3 = np.full(len(train), np.nan)

for fold_id, (tr_start, tr_end, val_start, val_end) in enumerate(fold_boundaries, start=1):
    train_mask = (train["timestamp"] >= tr_start) & (train["timestamp"] <= tr_end)
    val_mask   = (train["timestamp"] >= val_start) & (train["timestamp"] <= val_end)

    fold_train = train.loc[train_mask].copy()
    fold_val   = train.loc[val_mask].copy()
    fold_train, fold_val = add_v3_fold_features(fold_train, fold_val)

    X_train, y_train = fold_train[FEATURES_V3], fold_train[TARGET]
    X_val,   y_val    = fold_val[FEATURES_V3],   fold_val[TARGET]
    trend_train, trend_val = fold_train[["trend_idx"]], fold_val[["trend_idx"]]

    trend_model = LinearRegression().fit(trend_train, y_train)
    trend_pred_train = trend_model.predict(trend_train)
    trend_pred_val   = trend_model.predict(trend_val)
    resid_train = y_train - trend_pred_train

    xgb_model = xgb.XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1)
    xgb_model.fit(X_train, resid_train)
    resid_pred_val = xgb_model.predict(X_val)

    final_pred = trend_pred_val + resid_pred_val
    residual = y_val.values - final_pred

    fold_results_v3[fold_id] = {
        "xgb_model": xgb_model, "X_val": X_val, "y_val": y_val,
        "final_pred": final_pred, "residual": residual,
        "timestamps": fold_val["timestamp_central"].values,
    }
    oof_trend_xgb_v3[val_mask.values] = final_pred
    print(f"Fold {fold_id} RMSE: {mean_squared_error(y_val, final_pred) ** 0.5:.2f}")

# -------------------------
# Overall Metrics Evaluation
# -------------------------
valid_v3 = ~np.isnan(oof_trend_xgb_v3)
y_true_v3 = train.loc[valid_v3, TARGET]
y_pred_v3 = oof_trend_xgb_v3[valid_v3]

oof_rmse_v3 = mean_squared_error(y_true_v3, y_pred_v3) ** 0.5
oof_mape_v3 = mean_absolute_percentage_error(y_true_v3, y_pred_v3) * 100

print("-" * 40)
print(f"Trend + XGB + swing/storm proxy (v3), CV OOF RMSE: {oof_rmse_v3:.2f}   (v2: {oof_rmse_v2:.2f})")
print(f"Trend + XGB + swing/storm proxy (v3), CV OOF MAPE: {oof_mape_v3:.2f}%")

# -------------------------
# MLflow Logging
# -------------------------
with mlflow.start_run(run_name="Trend_XGB_v3"):
    # Log general parameters & experiment strategy
    mlflow.log_param("model_type", "LinearRegression_Trend + XGBoost_Residual")
    mlflow.log_param("experiment_version", "v3")
    mlflow.log_param("detrended", True)
    mlflow.log_param("features", "FEATURES_V3 (Extreme events, holiday interactions, temp_swing, high_precip)")
    mlflow.log_param("cv_strategy", "5-fold-expanding")
    
    # Log fold feature-engineering metadata
    mlflow.log_param("extreme_heat_quantile", 0.95)
    mlflow.log_param("extreme_cold_quantile", 0.05)
    mlflow.log_param("extreme_precip_quantile", 0.95)
    
    # Log XGBoost hyperparameters
    mlflow.log_param("n_estimators", 500)
    mlflow.log_param("learning_rate", 0.05)
    mlflow.log_param("max_depth", 6)
    
    # Log metrics
    mlflow.log_metric("cv_oof_rmse", oof_rmse_v3)
    mlflow.log_metric("cv_oof_mape", oof_mape_v3)
    
    print("Successfully logged params and metrics to MLflow (no model artifacts saved).")

Fold 1 RMSE: 409.10
Fold 2 RMSE: 380.09
Fold 3 RMSE: 386.20
Fold 4 RMSE: 452.64
Fold 5 RMSE: 381.94
----------------------------------------
Trend + XGB + swing/storm proxy (v3), CV OOF RMSE: 402.95   (v2: 423.57)
Trend + XGB + swing/storm proxy (v3), CV OOF MAPE: 3.49%
Successfully logged params and metrics to MLflow (no model artifacts saved).
🏃 View run Trend_XGB_v3 at: https://dagshub.com/dunnioluajayi/electricity-distribution-forecast.mlflow/#/experiments/2/runs/6120158c410e4567bf4987bd652f1528
🧪 View experiment at: https://dagshub.com/dunnioluajayi/electricity-distribution-forecast.mlflow/#/experiments/2


## LightGBM (v3 features)

In [13]:
fold_results_lgbm = {}
oof_trend_lgbm = np.full(len(train), np.nan)

for fold_id, (tr_start, tr_end, val_start, val_end) in enumerate(fold_boundaries, start=1):
    train_mask = (train["timestamp"] >= tr_start) & (train["timestamp"] <= tr_end)
    val_mask   = (train["timestamp"] >= val_start) & (train["timestamp"] <= val_end)

    fold_train = train.loc[train_mask].copy()
    fold_val   = train.loc[val_mask].copy()
    fold_train, fold_val = add_v3_fold_features(fold_train, fold_val)

    X_train, y_train = fold_train[FEATURES_V3], fold_train[TARGET]
    X_val,   y_val    = fold_val[FEATURES_V3],   fold_val[TARGET]
    trend_train, trend_val = fold_train[["trend_idx"]], fold_val[["trend_idx"]]

    trend_model = LinearRegression().fit(trend_train, y_train)
    trend_pred_train = trend_model.predict(trend_train)
    trend_pred_val   = trend_model.predict(trend_val)
    resid_train = y_train - trend_pred_train

    lgbm_model = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1, verbosity=-1)
    lgbm_model.fit(X_train, resid_train)
    resid_pred_val = lgbm_model.predict(X_val)

    final_pred = trend_pred_val + resid_pred_val
    residual = y_val.values - final_pred

    fold_results_lgbm[fold_id] = {
        "model": lgbm_model, "X_val": X_val, "y_val": y_val,
        "final_pred": final_pred, "residual": residual,
        "timestamps": fold_val["timestamp_central"].values,
    }
    oof_trend_lgbm[val_mask.values] = final_pred
    print(f"Fold {fold_id} RMSE: {mean_squared_error(y_val, final_pred) ** 0.5:.2f}")

# -------------------------
# Overall Metrics Evaluation
# -------------------------
valid_lgbm = ~np.isnan(oof_trend_lgbm)
y_true_lgbm = train.loc[valid_lgbm, TARGET]
y_pred_lgbm = oof_trend_lgbm[valid_lgbm]

oof_rmse_lgbm = mean_squared_error(y_true_lgbm, y_pred_lgbm) ** 0.5
oof_mape_lgbm = mean_absolute_percentage_error(y_true_lgbm, y_pred_lgbm) * 100

print("-" * 40)
print(f"Trend + LightGBM (v3 features), CV OOF RMSE: {oof_rmse_lgbm:.2f}   (XGB v3: {oof_rmse_v3:.2f})")
print(f"Trend + LightGBM (v3 features), CV OOF MAPE: {oof_mape_lgbm:.2f}%")

# -------------------------
# MLflow Logging
# -------------------------
with mlflow.start_run(run_name="Trend_LightGBM_v3"):
    # Log general parameters & experiment strategy
    mlflow.log_param("model_type", "LinearRegression_Trend + LightGBM_Residual")
    mlflow.log_param("experiment_version", "v3")
    mlflow.log_param("detrended", True)
    mlflow.log_param("features", "FEATURES_V3 (Extreme events, holiday interactions, temp_swing, high_precip)")
    mlflow.log_param("cv_strategy", "5-fold-expanding")
    
    # Log fold feature-engineering metadata
    mlflow.log_param("extreme_heat_quantile", 0.95)
    mlflow.log_param("extreme_cold_quantile", 0.05)
    mlflow.log_param("extreme_precip_quantile", 0.95)
    
    # Log LightGBM hyperparameters
    mlflow.log_param("n_estimators", 500)
    mlflow.log_param("learning_rate", 0.05)
    mlflow.log_param("max_depth", 6)
    
    # Log metrics
    mlflow.log_metric("cv_oof_rmse", oof_rmse_lgbm)
    mlflow.log_metric("cv_oof_mape", oof_mape_lgbm)
    
    print("Successfully logged params and metrics to MLflow (no model artifacts saved).")

Fold 1 RMSE: 404.38
Fold 2 RMSE: 385.28
Fold 3 RMSE: 388.04
Fold 4 RMSE: 428.33
Fold 5 RMSE: 384.30
----------------------------------------
Trend + LightGBM (v3 features), CV OOF RMSE: 398.44   (XGB v3: 402.95)
Trend + LightGBM (v3 features), CV OOF MAPE: 3.48%
Successfully logged params and metrics to MLflow (no model artifacts saved).
🏃 View run Trend_LightGBM_v3 at: https://dagshub.com/dunnioluajayi/electricity-distribution-forecast.mlflow/#/experiments/2/runs/42ce2ea4b3194a8d910cda1e4715402b
🧪 View experiment at: https://dagshub.com/dunnioluajayi/electricity-distribution-forecast.mlflow/#/experiments/2


## CatBoost (v3 features)

In [14]:
fold_results_cat = {}
oof_trend_cat = np.full(len(train), np.nan)

for fold_id, (tr_start, tr_end, val_start, val_end) in enumerate(fold_boundaries, start=1):
    train_mask = (train["timestamp"] >= tr_start) & (train["timestamp"] <= tr_end)
    val_mask   = (train["timestamp"] >= val_start) & (train["timestamp"] <= val_end)

    fold_train = train.loc[train_mask].copy()
    fold_val   = train.loc[val_mask].copy()
    fold_train, fold_val = add_v3_fold_features(fold_train, fold_val)

    X_train, y_train = fold_train[FEATURES_V3], fold_train[TARGET]
    X_val,   y_val    = fold_val[FEATURES_V3],   fold_val[TARGET]
    trend_train, trend_val = fold_train[["trend_idx"]], fold_val[["trend_idx"]]

    trend_model = LinearRegression().fit(trend_train, y_train)
    trend_pred_train = trend_model.predict(trend_train)
    trend_pred_val   = trend_model.predict(trend_val)
    resid_train = y_train - trend_pred_train

    cat_model = cb.CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6, random_state=42, thread_count=-1, verbose=0)
    cat_model.fit(X_train, resid_train)
    resid_pred_val = cat_model.predict(X_val)

    final_pred = trend_pred_val + resid_pred_val
    residual = y_val.values - final_pred

    fold_results_cat[fold_id] = {
        "model": cat_model, "X_val": X_val, "y_val": y_val,
        "final_pred": final_pred, "residual": residual,
        "timestamps": fold_val["timestamp_central"].values,
    }
    oof_trend_cat[val_mask.values] = final_pred
    print(f"Fold {fold_id} RMSE: {mean_squared_error(y_val, final_pred) ** 0.5:.2f}")

# -------------------------
# Overall Metrics Evaluation
# -------------------------
valid_cat = ~np.isnan(oof_trend_cat)
y_true_cat = train.loc[valid_cat, TARGET]
y_pred_cat = oof_trend_cat[valid_cat]

oof_rmse_cat = mean_squared_error(y_true_cat, y_pred_cat) ** 0.5
oof_mape_cat = mean_absolute_percentage_error(y_true_cat, y_pred_cat) * 100

print("-" * 40)
print(f"Trend + CatBoost (v3 features), CV OOF RMSE: {oof_rmse_cat:.2f}")
print(f"Trend + CatBoost (v3 features), CV OOF MAPE: {oof_mape_cat:.2f}%")

# -------------------------
# MLflow Logging
# -------------------------
with mlflow.start_run(run_name="Trend_CatBoost_v3"):
    # Log general parameters & experiment strategy
    mlflow.log_param("model_type", "LinearRegression_Trend + CatBoost_Residual")
    mlflow.log_param("experiment_version", "v3")
    mlflow.log_param("detrended", True)
    mlflow.log_param("features", "FEATURES_V3 (Extreme events, holiday interactions, temp_swing, high_precip)")
    mlflow.log_param("cv_strategy", "5-fold-expanding")
    
    # Log fold feature-engineering metadata
    mlflow.log_param("extreme_heat_quantile", 0.95)
    mlflow.log_param("extreme_cold_quantile", 0.05)
    mlflow.log_param("extreme_precip_quantile", 0.95)
    
    # Log CatBoost hyperparameters
    mlflow.log_param("iterations", 500)
    mlflow.log_param("learning_rate", 0.05)
    mlflow.log_param("depth", 6)
    
    # Log metrics
    mlflow.log_metric("cv_oof_rmse", oof_rmse_cat)
    mlflow.log_metric("cv_oof_mape", oof_mape_cat)
    
    print("Successfully logged params and metrics to MLflow (no model artifacts saved).")

Fold 1 RMSE: 399.87
Fold 2 RMSE: 367.75
Fold 3 RMSE: 381.41
Fold 4 RMSE: 440.00
Fold 5 RMSE: 378.68
----------------------------------------
Trend + CatBoost (v3 features), CV OOF RMSE: 394.39
Trend + CatBoost (v3 features), CV OOF MAPE: 3.46%
Successfully logged params and metrics to MLflow (no model artifacts saved).
🏃 View run Trend_CatBoost_v3 at: https://dagshub.com/dunnioluajayi/electricity-distribution-forecast.mlflow/#/experiments/2/runs/02bebb543beb4a0f86b59236eddf1a2e
🧪 View experiment at: https://dagshub.com/dunnioluajayi/electricity-distribution-forecast.mlflow/#/experiments/2


## KNN (v3 features)

In [15]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler

oof_trend_knn = np.full(len(train), np.nan)
fold_results_knn = {}

for fold_id, (tr_start, tr_end, val_start, val_end) in enumerate(fold_boundaries, start=1):
    train_mask = (train["timestamp"] >= tr_start) & (train["timestamp"] <= tr_end)
    val_mask   = (train["timestamp"] >= val_start) & (train["timestamp"] <= val_end)

    X_train, y_train = train.loc[train_mask, FEATURES], train.loc[train_mask, TARGET]
    X_val,   y_val    = train.loc[val_mask,   FEATURES], train.loc[val_mask,   TARGET]
    trend_train = train.loc[train_mask, ["trend_idx"]]
    trend_val   = train.loc[val_mask,   ["trend_idx"]]

    trend_model = LinearRegression()
    trend_model.fit(trend_train, y_train)
    trend_pred_train = trend_model.predict(trend_train)
    trend_pred_val   = trend_model.predict(trend_val)
    resid_train = y_train - trend_pred_train

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled   = scaler.transform(X_val)

    knn_model = KNeighborsRegressor(n_neighbors=15, weights="distance", n_jobs=-1)
    knn_model.fit(X_train_scaled, resid_train)
    resid_pred_val = knn_model.predict(X_val_scaled)

    final_pred = trend_pred_val + resid_pred_val
    residual = y_val.values - final_pred

    fold_results_knn[fold_id] = {
        "model": knn_model, "scaler": scaler, "X_val": X_val, "y_val": y_val,
        "final_pred": final_pred, "residual": residual,
        "timestamps": train.loc[val_mask, "timestamp_central"].values,
    }
    oof_trend_knn[val_mask.values] = final_pred
    print(f"Fold {fold_id} RMSE: {mean_squared_error(y_val, final_pred) ** 0.5:.2f}")

valid_knn = ~np.isnan(oof_trend_knn)
y_true_knn = train.loc[valid_knn, TARGET]
y_pred_knn = oof_trend_knn[valid_knn]


oof_rmse_knn = mean_squared_error(y_true_knn, y_pred_knn) ** 0.5
oof_mape_knn = mean_absolute_percentage_error(y_true_knn, y_pred_knn) * 100

print("-" * 40)
print(f"Trend + KNN (v3 features), CV OOF RMSE: {oof_rmse_knn:.2f}   (XGB v3: {oof_rmse_v3:.2f})")
print(f"Trend + LightGBM (v3 features), CV OOF MAPE: {oof_mape_knn:.2f}%")

# -------------------------
# MLflow Logging
# -------------------------
with mlflow.start_run(run_name="Trend_KNN "):
    # Log general parameters & experiment strategy
    mlflow.log_param("model_type", "LinearRegression_Trend + KNN")
    mlflow.log_param("experiment_version", "v3")
    mlflow.log_param("detrended", True)
    mlflow.log_param("features", "FEATURES_V3 (Extreme events, holiday interactions, temp_swing, high_precip)")
    mlflow.log_param("cv_strategy", "5-fold-expanding")
    
    # Log fold feature-engineering metadata
    mlflow.log_param("extreme_heat_quantile", 0.95)
    mlflow.log_param("extreme_cold_quantile", 0.05)
    mlflow.log_param("extreme_precip_quantile", 0.95)
    
    # Log KNN hyperparameters
    mlflow.log_param("n_neighbors", 15)
    mlflow.log_param("weights", "distance")
    
    
    # Log metrics
    mlflow.log_metric("cv_oof_rmse", oof_rmse_knn)
    mlflow.log_metric("cv_oof_mape", oof_mape_knn)
    
    print("Successfully logged params and metrics to MLflow (no model artifacts saved).")

Fold 1 RMSE: 551.46
Fold 2 RMSE: 495.20
Fold 3 RMSE: 540.93
Fold 4 RMSE: 611.66
Fold 5 RMSE: 560.85
----------------------------------------
Trend + KNN (v3 features), CV OOF RMSE: 553.32   (XGB v3: 402.95)
Trend + LightGBM (v3 features), CV OOF MAPE: 5.01%
Successfully logged params and metrics to MLflow (no model artifacts saved).
🏃 View run Trend_KNN  at: https://dagshub.com/dunnioluajayi/electricity-distribution-forecast.mlflow/#/experiments/2/runs/49e4cf8374394d79b2a919d41abe1bdd
🧪 View experiment at: https://dagshub.com/dunnioluajayi/electricity-distribution-forecast.mlflow/#/experiments/2


## Model Comparison — RMSE / MAPE / Peak-MAPE

In [16]:
def compute_metrics(y_true, y_pred, peak_pct=0.05):
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    threshold = np.quantile(y_true, 1 - peak_pct)
    peak_mask = y_true >= threshold
    peak_mape = np.mean(np.abs((y_true[peak_mask] - y_pred[peak_mask]) / y_true[peak_mask])) * 100
    return mape, peak_mape

model_rows = []
for name, oof, valid_mask in [
    ("Linear Regression", oof_lr, valid_lr),
    ("XGBoost v3", oof_trend_xgb_v3, valid_v3),
    ("LightGBM v3", oof_trend_lgbm, valid_lgbm),
    ("CatBoost v3", oof_trend_cat, valid_cat),
    ("KNN v3", oof_trend_knn, valid_knn),
]:
    y_true = train.loc[valid_mask, TARGET].values
    y_pred = oof[valid_mask]
    mape, peak_mape = compute_metrics(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    model_rows.append({"model": name, "rmse": rmse, "mape": mape, "peak_mape": peak_mape})

model_comparison = pd.DataFrame(model_rows).sort_values("peak_mape").reset_index(drop=True)
model_comparison

,model,rmse,mape,peak_mape
0,CatBoost v3,394.388962,3.458383,4.006626
1,LightGBM v3,398.436425,3.479007,4.015252
2,XGBoost v3,402.953178,3.486192,4.116590
3,KNN v3,553.316785,5.008921,4.288359
4,Linear Regression,762.385365,6.468600,5.756777


## Residual Correlation Check

In [17]:
resid_lr_arr  = train[TARGET].values - oof_lr
resid_xgb_arr = train[TARGET].values - oof_trend_xgb_v3
resid_lgbm_arr = train[TARGET].values - oof_trend_lgbm
resid_cat_arr = train[TARGET].values - oof_trend_cat
resid_knn_arr = train[TARGET].values - oof_trend_knn

common_valid = valid_lr & valid_v3 & valid_lgbm & valid_cat & valid_knn

corr_df = pd.DataFrame({
    "resid_lr":   resid_lr_arr[common_valid],
    "resid_xgb":  resid_xgb_arr[common_valid],
    "resid_lgbm": resid_lgbm_arr[common_valid],
    "resid_cat":  resid_cat_arr[common_valid],
    "resid_knn":  resid_knn_arr[common_valid],
})

print(f"Rows compared: {common_valid.sum():,}")
corr_df.corr()

Rows compared: 43,824


,resid_lr,resid_xgb,resid_lgbm,resid_cat,resid_knn
resid_lr,1.000000,0.441079,0.448766,0.482669,0.575651
resid_xgb,0.441079,1.000000,0.962972,0.913331,0.657815
resid_lgbm,0.448766,0.962972,1.000000,0.924441,0.668251
resid_cat,0.482669,0.913331,0.924441,1.000000,0.708256
resid_knn,0.575651,0.657815,0.668251,0.708256,1.000000


## Hill-Climbing Ensemble

In [18]:
def hill_climb_ensemble(oof_dict, y_true_full, valid_mask, n_iterations=50, tol=1e-6):
    preds = {name: arr[valid_mask] for name, arr in oof_dict.items()}
    y = y_true_full[valid_mask]

    solo_rmse = {name: mean_squared_error(y, p) ** 0.5 for name, p in preds.items()}
    best_start = min(solo_rmse, key=solo_rmse.get)

    selected = [best_start]
    ensemble_preds = preds[best_start].copy()
    history = [solo_rmse[best_start]]

    for _ in range(n_iterations):
        best_rmse, best_name, best_preds = np.inf, None, None
        for name, p in preds.items():
            candidate = (ensemble_preds * len(selected) + p) / (len(selected) + 1)
            rmse = mean_squared_error(y, candidate) ** 0.5
            if rmse < best_rmse:
                best_rmse, best_name, best_preds = rmse, name, candidate
        if best_rmse < history[-1] - tol:
            selected.append(best_name)
            ensemble_preds = best_preds
            history.append(best_rmse)
        else:
            break

    weights = pd.Series(selected).value_counts(normalize=True).sort_values(ascending=False)
    return ensemble_preds, selected, weights, history, solo_rmse

oof_dict = {
    "lr": oof_lr, "xgb_v3": oof_trend_xgb_v3,
    "lgbm": oof_trend_lgbm, "catboost": oof_trend_cat, "knn": oof_trend_knn,
}
y_true_full = train[TARGET].values

ensemble_preds, selected, ensemble_weights, ensemble_history, solo_rmse = hill_climb_ensemble(
    oof_dict, y_true_full, common_valid
)

print("Solo RMSE (on common_valid rows):")
for name, rmse in sorted(solo_rmse.items(), key=lambda x: x[1]):
    print(f"  {name}: {rmse:.2f}")

print(f"\nHill-climb ensemble OOF RMSE: {ensemble_history[-1]:.2f}")
print("\nModel weights (by selection frequency):")
print(ensemble_weights)

Solo RMSE (on common_valid rows):
  catboost: 394.39
  lgbm: 398.44
  xgb_v3: 402.95
  knn: 553.32
  lr: 762.39

Hill-climb ensemble OOF RMSE: 388.50

Model weights (by selection frequency):
catboost    0.5
lgbm        0.5
Name: proportion, dtype: float64


## Hyperparameter Tuning — Shared Harness

In [19]:
def oof_rmse_for_model(model_fn, features=None):
    features = features or FEATURES_V3
    oof_trial = np.full(len(train), np.nan)
    for fold_id, (tr_start, tr_end, val_start, val_end) in enumerate(fold_boundaries, start=1):
        train_mask = (train["timestamp"] >= tr_start) & (train["timestamp"] <= tr_end)
        val_mask   = (train["timestamp"] >= val_start) & (train["timestamp"] <= val_end)

        fold_train = train.loc[train_mask].copy()
        fold_val   = train.loc[val_mask].copy()
        fold_train, fold_val = add_v3_fold_features(fold_train, fold_val)

        X_train, y_train = fold_train[features], fold_train[TARGET]
        X_val,   y_val    = fold_val[features],   fold_val[TARGET]
        trend_train, trend_val = fold_train[["trend_idx"]], fold_val[["trend_idx"]]

        trend_model = LinearRegression().fit(trend_train, y_train)
        resid_train = y_train - trend_model.predict(trend_train)
        trend_pred_val = trend_model.predict(trend_val)

        model = model_fn()
        model.fit(X_train, resid_train)
        resid_pred_val = model.predict(X_val)

        oof_trial[val_mask.values] = trend_pred_val + resid_pred_val

    valid_trial = ~np.isnan(oof_trial)
    return mean_squared_error(train.loc[valid_trial, TARGET], oof_trial[valid_trial]) ** 0.5

## Hyperparameter Tuning — XGBoost (Optuna)

In [20]:
def objective_xgb(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 800),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10, log=True),
    }
    model_fn = lambda: xgb.XGBRegressor(**params, random_state=42, n_jobs=-1)
    return oof_rmse_for_model(model_fn)

study_xgb = optuna.create_study(direction="minimize", study_name="xgb_v3_tuning")
study_xgb.optimize(objective_xgb, n_trials=40, show_progress_bar=True)
print(f"Best XGB v3 OOF RMSE: {study_xgb.best_value:.2f}  (default-params baseline: {oof_rmse_v3:.2f})")
print(study_xgb.best_params)

Best trial: 10. Best value: 391.323: 100%|██████████| 40/40 [11:58<00:00, 17.95s/it]

Best XGB v3 OOF RMSE: 391.32  (default-params baseline: 402.95)
{'n_estimators': 768, 'learning_rate': 0.027514166028504757, 'max_depth': 6, 'min_child_weight': 20, 'subsample': 0.6056574296988497, 'colsample_bytree': 0.7324168634265418, 'reg_alpha': 0.028070735080041896, 'reg_lambda': 0.03139039480398197}


## Hyperparameter Tuning — LightGBM (Optuna)

In [21]:
def objective_lgbm(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 800),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 16, 128),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10, log=True),
    }
    model_fn = lambda: lgb.LGBMRegressor(**params, random_state=42, n_jobs=-1, verbosity=-1)
    return oof_rmse_for_model(model_fn)

study_lgbm = optuna.create_study(direction="minimize", study_name="lgbm_v3_tuning")
study_lgbm.optimize(objective_lgbm, n_trials=40, show_progress_bar=True)
print(f"Best LightGBM OOF RMSE: {study_lgbm.best_value:.2f}  (default-params baseline: {oof_rmse_lgbm:.2f})")
print(study_lgbm.best_params)

Best trial: 12. Best value: 391.14: 100%|██████████| 40/40 [09:17<00:00, 13.94s/it] 

Best LightGBM OOF RMSE: 391.14  (default-params baseline: 398.44)
{'n_estimators': 660, 'learning_rate': 0.07401535558645038, 'num_leaves': 16, 'max_depth': 8, 'min_child_samples': 45, 'subsample': 0.8243728393479369, 'colsample_bytree': 0.6050248151198275, 'reg_alpha': 0.06841420888309284, 'reg_lambda': 0.0010025119167274652}


## Hyperparameter Tuning — CatBoost (Optuna)

In [22]:
def objective_cat(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 200, 800),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "depth": trial.suggest_int("depth", 3, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-2, 10, log=True),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
        "random_strength": trial.suggest_float("random_strength", 1e-3, 10, log=True),
        "border_count": trial.suggest_int("border_count", 32, 255),
    }
    model_fn = lambda: cb.CatBoostRegressor(**params, random_state=42, thread_count=-1, verbose=0)
    return oof_rmse_for_model(model_fn)

study_cat = optuna.create_study(direction="minimize", study_name="cat_v3_tuning")
study_cat.optimize(objective_cat, n_trials=40, show_progress_bar=True)
print(f"Best CatBoost OOF RMSE: {study_cat.best_value:.2f}  (default-params baseline: {oof_rmse_cat:.2f})")
print(study_cat.best_params)

Best trial: 39. Best value: 387.799: 100%|██████████| 40/40 [11:51<00:00, 17.80s/it]

Best CatBoost OOF RMSE: 387.80  (default-params baseline: 394.39)
{'iterations': 670, 'learning_rate': 0.047125122732693765, 'depth': 7, 'l2_leaf_reg': 1.0262766725860368, 'bagging_temperature': 0.5281239408590017, 'random_strength': 1.7562764424557915, 'border_count': 151}


## Tuned Model Comparison

In [23]:
tuned_comparison = pd.DataFrame([
    {"model": "xgb",      "tuned_oof_rmse": study_xgb.best_value},
    {"model": "lgbm",     "tuned_oof_rmse": study_lgbm.best_value},
    {"model": "catboost", "tuned_oof_rmse": study_cat.best_value},
]).sort_values("tuned_oof_rmse").reset_index(drop=True)
tuned_comparison

,model,tuned_oof_rmse
0,catboost,387.799134
1,lgbm,391.139764
2,xgb,391.322605


## Target-Encoding Check — Holiday Identity

In [25]:
cal = USFederalHolidayCalendar()
start = df["timestamp_central"].min().tz_localize(None)
end   = df["timestamp_central"].max().tz_localize(None)
holiday_names_map = cal.holidays(start=start, end=end, return_name=True)

df["holiday_name"] = df["timestamp_central"].dt.normalize().dt.tz_localize(None).map(holiday_names_map)

train = df[df["timestamp"] <= TRAIN_END].reset_index(drop=True)
test  = df[df["timestamp"] >= TEST_START].reset_index(drop=True)

In [26]:
def add_holiday_freq(fold_train, fold_val):
    freq_map = fold_train["holiday_name"].value_counts(normalize=True)
    for part in (fold_train, fold_val):
        part["holiday_freq"] = part["holiday_name"].map(freq_map).fillna(0.0)
    return fold_train, fold_val

def oof_rmse_for_model_holiday(model_fn):
    features = FEATURES_V3 + ["holiday_freq"]
    oof_trial = np.full(len(train), np.nan)
    for fold_id, (tr_start, tr_end, val_start, val_end) in enumerate(fold_boundaries, start=1):
        train_mask = (train["timestamp"] >= tr_start) & (train["timestamp"] <= tr_end)
        val_mask   = (train["timestamp"] >= val_start) & (train["timestamp"] <= val_end)

        fold_train = train.loc[train_mask].copy()
        fold_val   = train.loc[val_mask].copy()
        fold_train, fold_val = add_v3_fold_features(fold_train, fold_val)
        fold_train, fold_val = add_holiday_freq(fold_train, fold_val)

        X_train, y_train = fold_train[features], fold_train[TARGET]
        X_val,   y_val    = fold_val[features],   fold_val[TARGET]
        trend_train, trend_val = fold_train[["trend_idx"]], fold_val[["trend_idx"]]

        trend_model = LinearRegression().fit(trend_train, y_train)
        resid_train = y_train - trend_model.predict(trend_train)
        trend_pred_val = trend_model.predict(trend_val)

        model = model_fn()
        model.fit(X_train, resid_train)
        resid_pred_val = model.predict(X_val)

        oof_trial[val_mask.values] = trend_pred_val + resid_pred_val

    valid_trial = ~np.isnan(oof_trial)
    return mean_squared_error(train.loc[valid_trial, TARGET], oof_trial[valid_trial]) ** 0.5

MODEL_BUILDERS = {
    "xgb":      lambda params: xgb.XGBRegressor(**params, random_state=42, n_jobs=-1),
    "lgbm":     lambda params: lgb.LGBMRegressor(**params, random_state=42, n_jobs=-1, verbosity=-1),
    "catboost": lambda params: cb.CatBoostRegressor(**params, random_state=42, thread_count=-1, verbose=0),
}

CHAMPION_NAME = tuned_comparison.iloc[0]["model"]
CHAMPION_OOF_RMSE = tuned_comparison.iloc[0]["tuned_oof_rmse"]
CHAMPION_PARAMS = {"xgb": study_xgb.best_params, "lgbm": study_lgbm.best_params, "catboost": study_cat.best_params}[CHAMPION_NAME]

champion_model_fn = lambda: MODEL_BUILDERS[CHAMPION_NAME](CHAMPION_PARAMS)
holiday_oof_rmse = oof_rmse_for_model_holiday(champion_model_fn)

print(f"Champion ({CHAMPION_NAME}) OOF RMSE without holiday_freq: {CHAMPION_OOF_RMSE:.2f}")
print(f"Champion ({CHAMPION_NAME}) OOF RMSE with holiday_freq:    {holiday_oof_rmse:.2f}")

USE_HOLIDAY_FEATURE = holiday_oof_rmse < CHAMPION_OOF_RMSE
print(f"\nUse holiday_freq in final model: {USE_HOLIDAY_FEATURE}")

Champion (catboost) OOF RMSE without holiday_freq: 387.80
Champion (catboost) OOF RMSE with holiday_freq:    387.78

Use holiday_freq in final model: True


## Champion Selection & Final Model Gate

In [27]:
WINNING_FEATURES = FEATURES_V3 + ["holiday_freq"] if USE_HOLIDAY_FEATURE else FEATURES_V3

train_final = train.copy()
test_final = test.copy()

heat_thresh   = train_final["tmax"].quantile(0.95)
cold_thresh   = train_final["tmin"].quantile(0.05)
precip_thresh = train_final["precip_mm"].quantile(0.95)
for part in (train_final, test_final):
    part["is_extreme_heat_event"] = (part["tmax"] > heat_thresh).astype(int)
    part["is_extreme_cold_event"] = (part["tmin"] < cold_thresh).astype(int)
    part["is_holiday_x_extreme"] = (
        part["is_holiday"] * (part["is_extreme_heat_event"] | part["is_extreme_cold_event"])
    ).astype(int)
    part["is_high_precip_event"] = (part["precip_mm"] > precip_thresh).astype(int)

if USE_HOLIDAY_FEATURE:
    freq_map = train_final["holiday_name"].value_counts(normalize=True)
    for part in (train_final, test_final):
        part["holiday_freq"] = part["holiday_name"].map(freq_map).fillna(0.0)

X_train_final, y_train_final = train_final[WINNING_FEATURES], train_final[TARGET]
X_test_final,  y_test_final  = test_final[WINNING_FEATURES],  test_final[TARGET]

trend_model_final = LinearRegression().fit(train_final[["trend_idx"]], y_train_final)
trend_train_final = trend_model_final.predict(train_final[["trend_idx"]])
trend_test_final  = trend_model_final.predict(test_final[["trend_idx"]])

residual_target_final = y_train_final - trend_train_final

final_model = MODEL_BUILDERS[CHAMPION_NAME](CHAMPION_PARAMS)
final_model.fit(X_train_final, residual_target_final)
pred_test_final = trend_test_final + final_model.predict(X_test_final)

test_rmse = mean_squared_error(y_test_final, pred_test_final) ** 0.5
test_mape = np.mean(np.abs((y_test_final.to_numpy() - pred_test_final) / y_test_final.to_numpy())) * 100

print(f"Champion model: {CHAMPION_NAME}  |  Features: {'v3 + holiday_freq' if USE_HOLIDAY_FEATURE else 'v3'}")
print(f"Final test RMSE: {test_rmse:.2f}")
print(f"Final test MAPE: {test_mape:.3f}%")

Champion model: catboost  |  Features: v3 + holiday_freq
Final test RMSE: 351.70
Final test MAPE: 2.870%


## MLflow Logging — Production Candidate

In [29]:
artifact_dir = Path("artifacts")
artifact_dir.mkdir(exist_ok=True)
model_path = artifact_dir / "electricity_load_forecaster.pkl"

production_bundle = {
    "trend_model": trend_model_final,
    "residual_model": final_model,
    "features": WINNING_FEATURES,
    "target": TARGET,
    "model_family": f"trend_plus_tuned_{CHAMPION_NAME}",
    "feature_version": "v3_holiday" if USE_HOLIDAY_FEATURE else "v3",
    "train_end": str(train_final["timestamp"].max()),
    "test_start": str(test_final["timestamp"].min()),
}

joblib.dump(production_bundle, model_path)

with mlflow.start_run(run_name="production_candidate_final"):
    mlflow.log_params({
        "model_family": production_bundle["model_family"],
        "feature_version": production_bundle["feature_version"],
        **{f"{CHAMPION_NAME}_{k}": v for k, v in CHAMPION_PARAMS.items()},
        "train_rows": len(train_final),
        "test_rows": len(test_final),
        "purge_days": PURGE_DAYS,
        "train_end": str(train_final["timestamp"].max()),
        "test_start": str(test_final["timestamp"].min()),
    })

    mlflow.log_metrics({
        "cv_oof_rmse": float(CHAMPION_OOF_RMSE),
        "test_rmse": float(test_rmse),
        "test_mape_pct": float(test_mape),
        "test_peak_mape_pct": float(peak_mape),
    })

    mlflow.log_artifact(str(model_path), artifact_path="model")

print(f"Logged production candidate: {model_path}")

🏃 View run production_candidate_final at: https://dagshub.com/dunnioluajayi/electricity-distribution-forecast.mlflow/#/experiments/2/runs/11cd36785ea94acdb632dae84861671b
🧪 View experiment at: https://dagshub.com/dunnioluajayi/electricity-distribution-forecast.mlflow/#/experiments/2
Logged production candidate: artifacts/electricity_load_forecaster.pkl
